In [2]:
import os

DATASET_ROOT = "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset"

print("=" * 70)
print("🔍 RPC DATASET CHECK")
print("=" * 70)

if not os.path.exists(DATASET_ROOT):
    print("❌ Dataset path does not exist!")
else:
    print("✅ Dataset found!")
    print("\nContents:\n")

    for item in sorted(os.listdir(DATASET_ROOT)):
        path = os.path.join(DATASET_ROOT, item)

        if os.path.isdir(path):
            try:
                count = len(os.listdir(path))
            except:
                count = "?"

            print(f"📁 {item:<35} {count} files")
        else:
            size = os.path.getsize(path) / (1024**2)
            print(f"📄 {item:<35} {size:.2f} MB")

🔍 RPC DATASET CHECK
✅ Dataset found!

Contents:

📄 instances_test2019.json             53.38 MB
📄 instances_train2019.json            14.12 MB
📄 instances_val2019.json              13.39 MB
📁 retail_product_checkout             6 files
📁 test2019                            24000 files
📁 train2019                           53739 files
📁 val2019                             6000 files


In [6]:
# ============================================================
# RPC DATASET
# RANDOM 4000 TRAIN / 2000 TEST SPLIT
# ============================================================

import os
import json
import random
import shutil
from pathlib import Path

import yaml


# ============================================================
# 1. CONFIGURATION
# ============================================================

# 🔄 CHANGE THIS NUMBER FOR A DIFFERENT RANDOM SPLIT
RANDOM_SEED = 42

TRAIN_COUNT = 4000
TEST_COUNT = 2000

# Correct RPC paths
SOURCE_IMAGES = Path(
    "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/val2019"
)

SOURCE_JSON = Path(
    "/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/instances_val2019.json"
)

# Working directory
WORK_DIR = Path("/kaggle/working/rpc_4000_2000")

TRAIN_IMAGES = WORK_DIR / "images" / "train"
TEST_IMAGES = WORK_DIR / "images" / "test"

TRAIN_LABELS = WORK_DIR / "labels" / "train"
TEST_LABELS = WORK_DIR / "labels" / "test"

# ============================================================
# 2. CHECK DATASET
# ============================================================

print("=" * 70)
print("🔍 RPC DATASET CHECK")
print("=" * 70)

print("Source images:")
print(SOURCE_IMAGES)
print("Exists:", SOURCE_IMAGES.exists())

print()

print("Annotation file:")
print(SOURCE_JSON)
print("Exists:", SOURCE_JSON.exists())

if not SOURCE_IMAGES.exists():
    raise FileNotFoundError(
        f"❌ Images not found:\n{SOURCE_IMAGES}"
    )

if not SOURCE_JSON.exists():
    raise FileNotFoundError(
        f"❌ Annotation file not found:\n{SOURCE_JSON}"
    )


# ============================================================
# 3. FIND ALL 6000 VALIDATION IMAGES
# ============================================================

all_images = sorted([
    p for p in SOURCE_IMAGES.iterdir()
    if p.is_file()
    and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

print()
print("=" * 70)
print("📊 SOURCE DATASET")
print("=" * 70)

print("Total images found:", len(all_images))

if len(all_images) != 6000:
    print("⚠️ WARNING: Expected 6000 images.")


# ============================================================
# 4. RANDOM SPLIT
# ============================================================

if TRAIN_COUNT + TEST_COUNT > len(all_images):
    raise ValueError(
        "❌ Train + Test count is greater than available images."
    )

random.seed(RANDOM_SEED)

shuffled_images = all_images.copy()
random.shuffle(shuffled_images)

train_images = shuffled_images[:TRAIN_COUNT]
test_images = shuffled_images[
    TRAIN_COUNT:TRAIN_COUNT + TEST_COUNT
]

print()
print("=" * 70)
print("🎲 RANDOM SPLIT")
print("=" * 70)

print("Random seed :", RANDOM_SEED)
print("Train       :", len(train_images))
print("Test        :", len(test_images))


# ============================================================
# 5. CREATE DIRECTORIES
# ============================================================

for directory in [
    TRAIN_IMAGES,
    TEST_IMAGES,
    TRAIN_LABELS,
    TEST_LABELS
]:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# 6. COPY IMAGES
# ============================================================

print()
print("=" * 70)
print("📁 COPYING TRAIN IMAGES")
print("=" * 70)

for i, image_path in enumerate(train_images, 1):

    destination = TRAIN_IMAGES / image_path.name

    shutil.copy2(
        image_path,
        destination
    )

    if i % 500 == 0 or i == TRAIN_COUNT:
        print(f"Train: {i}/{TRAIN_COUNT}")


print()
print("=" * 70)
print("📁 COPYING TEST IMAGES")
print("=" * 70)

for i, image_path in enumerate(test_images, 1):

    destination = TEST_IMAGES / image_path.name

    shutil.copy2(
        image_path,
        destination
    )

    if i % 500 == 0 or i == TEST_COUNT:
        print(f"Test : {i}/{TEST_COUNT}")


print("\n✅ Image copying complete")


# ============================================================
# 7. VERIFY NO OVERLAP
# ============================================================

train_names = set(
    p.name for p in train_images
)

test_names = set(
    p.name for p in test_images
)

overlap = train_names.intersection(test_names)

print()
print("=" * 70)
print("🔎 SPLIT VERIFICATION")
print("=" * 70)

print("Train images :", len(train_names))
print("Test images  :", len(test_names))
print("Overlap      :", len(overlap))

if len(overlap) > 0:

    raise RuntimeError(
        f"❌ DATA LEAKAGE! {len(overlap)} images are in both sets."
    )

print("✅ NO IMAGE OVERLAP")


# ============================================================
# 8. LOAD COCO ANNOTATIONS
# ============================================================

print()
print("=" * 70)
print("📝 LOADING COCO ANNOTATIONS")
print("=" * 70)

with open(SOURCE_JSON, "r") as f:
    coco = json.load(f)

print("COCO images      :", len(coco["images"]))
print("COCO annotations :", len(coco["annotations"]))
print("Classes          :", len(coco["categories"]))


# ============================================================
# 9. CREATE IMAGE MAPPINGS
# ============================================================

image_info_by_id = {
    img["id"]: img
    for img in coco["images"]
}

filename_to_image_id = {
    img["file_name"]: img["id"]
    for img in coco["images"]
}


# ============================================================
# 10. CREATE CLASS MAPPING
# ============================================================

categories = sorted(
    coco["categories"],
    key=lambda x: x["id"]
)

category_id_to_yolo = {
    category["id"]: index
    for index, category in enumerate(categories)
}

class_names = [
    category["name"]
    for category in categories
]


# ============================================================
# 11. GROUP ANNOTATIONS BY IMAGE
# ============================================================

annotations_by_image = {}

for annotation in coco["annotations"]:

    image_id = annotation["image_id"]

    if image_id not in annotations_by_image:
        annotations_by_image[image_id] = []

    annotations_by_image[image_id].append(
        annotation
    )


# ============================================================
# 12. COCO → YOLO CONVERSION FUNCTION
# ============================================================

def convert_to_yolo(
    image_list,
    output_label_dir
):

    created_labels = 0
    total_boxes = 0
    missing_images = 0

    for image_path in image_list:

        filename = image_path.name

        if filename not in filename_to_image_id:

            print(
                f"⚠️ Missing COCO image entry: {filename}"
            )

            missing_images += 1
            continue

        image_id = filename_to_image_id[filename]

        image_info = image_info_by_id[image_id]

        img_width = image_info["width"]
        img_height = image_info["height"]

        annotations = annotations_by_image.get(
            image_id,
            []
        )

        label_path = (
            output_label_dir /
            f"{image_path.stem}.txt"
        )

        with open(label_path, "w") as f:

            for annotation in annotations:

                x, y, w, h = annotation["bbox"]

                # COCO → YOLO
                x_center = (
                    x + w / 2
                ) / img_width

                y_center = (
                    y + h / 2
                ) / img_height

                width = w / img_width
                height = h / img_height

                class_id = category_id_to_yolo[
                    annotation["category_id"]
                ]

                f.write(
                    f"{class_id} "
                    f"{x_center:.6f} "
                    f"{y_center:.6f} "
                    f"{width:.6f} "
                    f"{height:.6f}\n"
                )

                total_boxes += 1

        created_labels += 1

    return (
        created_labels,
        total_boxes,
        missing_images
    )


# ============================================================
# 13. CONVERT TRAIN LABELS
# ============================================================

print()
print("=" * 70)
print("📝 CONVERTING TRAIN ANNOTATIONS")
print("=" * 70)

train_created, train_boxes, train_missing = convert_to_yolo(
    train_images,
    TRAIN_LABELS
)

print("Label files     :", train_created)
print("Bounding boxes  :", train_boxes)
print("Missing entries :", train_missing)


# ============================================================
# 14. CONVERT TEST LABELS
# ============================================================

print()
print("=" * 70)
print("📝 CONVERTING TEST ANNOTATIONS")
print("=" * 70)

test_created, test_boxes, test_missing = convert_to_yolo(
    test_images,
    TEST_LABELS
)

print("Label files     :", test_created)
print("Bounding boxes  :", test_boxes)
print("Missing entries :", test_missing)


# ============================================================
# 15. CREATE data.yaml
# ============================================================

data_yaml = {
    "path": str(WORK_DIR),
    "train": "images/train",
    "val": "images/test",
    "nc": len(class_names),
    "names": class_names
}

yaml_path = WORK_DIR / "data.yaml"

with open(yaml_path, "w") as f:

    yaml.dump(
        data_yaml,
        f,
        sort_keys=False
    )


print()
print("=" * 70)
print("📄 DATA.YAML CREATED")
print("=" * 70)

print("Location:", yaml_path)
print("Classes :", len(class_names))


# ============================================================
# 16. FINAL DATASET VERIFICATION
# ============================================================

train_image_count = len(
    list(TRAIN_IMAGES.glob("*"))
)

test_image_count = len(
    list(TEST_IMAGES.glob("*"))
)

train_label_count = len(
    list(TRAIN_LABELS.glob("*.txt"))
)

test_label_count = len(
    list(TEST_LABELS.glob("*.txt"))
)

print()
print("=" * 70)
print("✅ FINAL DATASET CHECK")
print("=" * 70)

print()
print("TRAIN")
print("Images :", train_image_count)
print("Labels :", train_label_count)

print()
print("TEST")
print("Images :", test_image_count)
print("Labels :", test_label_count)

print()
print("EXPECTED")
print("Train images :", TRAIN_COUNT)
print("Test images  :", TEST_COUNT)

# ============================================================
# 17. FINAL RESULT
# ============================================================

if (
    train_image_count == TRAIN_COUNT
    and test_image_count == TEST_COUNT
    and train_label_count == TRAIN_COUNT
    and test_label_count == TEST_COUNT
):

    print()
    print("🎉 DATASET IS READY!")

else:

    print()
    print("⚠️ SOMETHING IS WRONG — CHECK THE COUNTS")


print()
print("=" * 70)
print("💾 FINAL LOCATION")
print("=" * 70)

print(WORK_DIR)
print()
print("data.yaml:")
print(yaml_path)

🔍 RPC DATASET CHECK
Source images:
/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/val2019
Exists: True

Annotation file:
/kaggle/input/datasets/diyer22/retail-product-checkout-dataset/instances_val2019.json
Exists: True

📊 SOURCE DATASET
Total images found: 6000

🎲 RANDOM SPLIT
Random seed : 42
Train       : 4000
Test        : 2000

📁 COPYING TRAIN IMAGES
Train: 500/4000
Train: 1000/4000
Train: 1500/4000
Train: 2000/4000
Train: 2500/4000
Train: 3000/4000
Train: 3500/4000
Train: 4000/4000

📁 COPYING TEST IMAGES
Test : 500/2000
Test : 1000/2000
Test : 1500/2000
Test : 2000/2000

✅ Image copying complete

🔎 SPLIT VERIFICATION
Train images : 4000
Test images  : 2000
Overlap      : 0
✅ NO IMAGE OVERLAP

📝 LOADING COCO ANNOTATIONS
COCO images      : 6000
COCO annotations : 73602
Classes          : 200

📝 CONVERTING TRAIN ANNOTATIONS
Label files     : 4000
Bounding boxes  : 48895
Missing entries : 0

📝 CONVERTING TEST ANNOTATIONS
Label files     : 2000
Bounding boxes  : 24707


In [7]:
# ============================================================
# VERIFY RANDOM SPLIT + CONVERT ANNOTATIONS
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

WORK_DIR = Path("/kaggle/working/rpc_4000_2000")

TRAIN_IMAGES = WORK_DIR / "images" / "train"
TEST_IMAGES  = WORK_DIR / "images" / "test"

TRAIN_LABELS = WORK_DIR / "labels" / "train"
TEST_LABELS  = WORK_DIR / "labels" / "test"

SOURCE_JSON = Path(
    "/kaggle/input/datasets/diyer22/"
    "retail-product-checkout-dataset/"
    "instances_val2019.json"
)

# ------------------------------------------------------------
# 1. CHECK IMAGE SPLIT
# ------------------------------------------------------------

train_files = {
    p.name for p in TRAIN_IMAGES.iterdir()
    if p.is_file()
}

test_files = {
    p.name for p in TEST_IMAGES.iterdir()
    if p.is_file()
}

overlap = train_files & test_files

print("=" * 70)
print("🔎 RANDOM SPLIT VERIFICATION")
print("=" * 70)

print("Train images :", len(train_files))
print("Test images  :", len(test_files))
print("Overlap      :", len(overlap))

if len(overlap) == 0:
    print("✅ NO TRAIN/TEST OVERLAP")
else:
    raise RuntimeError(
        f"❌ ERROR: {len(overlap)} images appear in both sets!"
    )

# ------------------------------------------------------------
# 2. LOAD COCO JSON
# ------------------------------------------------------------

print()
print("=" * 70)
print("📝 LOADING COCO ANNOTATIONS")
print("=" * 70)

with open(SOURCE_JSON, "r") as f:
    coco = json.load(f)

print("COCO images      :", len(coco["images"]))
print("COCO annotations :", len(coco["annotations"]))
print("Classes          :", len(coco["categories"]))

# ------------------------------------------------------------
# 3. CREATE MAPPINGS
# ------------------------------------------------------------

image_info = {
    img["id"]: img
    for img in coco["images"]
}

filename_to_id = {
    img["file_name"]: img["id"]
    for img in coco["images"]
}

categories = sorted(
    coco["categories"],
    key=lambda x: x["id"]
)

category_to_yolo = {
    cat["id"]: i
    for i, cat in enumerate(categories)
}

class_names = [
    cat["name"]
    for cat in categories
]

# ------------------------------------------------------------
# 4. GROUP ANNOTATIONS BY IMAGE
# ------------------------------------------------------------

annotations_by_image = {}

for ann in coco["annotations"]:

    image_id = ann["image_id"]

    if image_id not in annotations_by_image:
        annotations_by_image[image_id] = []

    annotations_by_image[image_id].append(ann)

# ------------------------------------------------------------
# 5. CONVERSION FUNCTION
# ------------------------------------------------------------

def convert_annotations(image_names, output_dir):

    created = 0
    boxes = 0
    missing = 0

    for filename in image_names:

        if filename not in filename_to_id:
            print("⚠️ Missing:", filename)
            missing += 1
            continue

        image_id = filename_to_id[filename]

        info = image_info[image_id]

        img_w = info["width"]
        img_h = info["height"]

        anns = annotations_by_image.get(
            image_id,
            []
        )

        label_path = output_dir / (
            Path(filename).stem + ".txt"
        )

        with open(label_path, "w") as f:

            for ann in anns:

                x, y, w, h = ann["bbox"]

                # COCO → YOLO
                xc = (x + w / 2) / img_w
                yc = (y + h / 2) / img_h

                wn = w / img_w
                hn = h / img_h

                class_id = category_to_yolo[
                    ann["category_id"]
                ]

                f.write(
                    f"{class_id} "
                    f"{xc:.6f} "
                    f"{yc:.6f} "
                    f"{wn:.6f} "
                    f"{hn:.6f}\n"
                )

                boxes += 1

        created += 1

    return created, boxes, missing

# ------------------------------------------------------------
# 6. TRAIN LABELS
# ------------------------------------------------------------

print()
print("=" * 70)
print("📝 CONVERTING TRAIN ANNOTATIONS")
print("=" * 70)

train_created, train_boxes, train_missing = convert_annotations(
    train_files,
    TRAIN_LABELS
)

print("Label files     :", train_created)
print("Bounding boxes  :", train_boxes)
print("Missing images  :", train_missing)

# ------------------------------------------------------------
# 7. TEST LABELS
# ------------------------------------------------------------

print()
print("=" * 70)
print("📝 CONVERTING TEST ANNOTATIONS")
print("=" * 70)

test_created, test_boxes, test_missing = convert_annotations(
    test_files,
    TEST_LABELS
)

print("Label files     :", test_created)
print("Bounding boxes  :", test_boxes)
print("Missing images  :", test_missing)

# ------------------------------------------------------------
# 8. VERIFY LABEL COUNTS
# ------------------------------------------------------------

train_label_count = len(
    list(TRAIN_LABELS.glob("*.txt"))
)

test_label_count = len(
    list(TEST_LABELS.glob("*.txt"))
)

print()
print("=" * 70)
print("📊 FINAL ANNOTATION CHECK")
print("=" * 70)

print("TRAIN")
print("Images :", len(train_files))
print("Labels :", train_label_count)
print("Boxes  :", train_boxes)

print()
print("TEST")
print("Images :", len(test_files))
print("Labels :", test_label_count)
print("Boxes  :", test_boxes)

# ------------------------------------------------------------
# 9. CREATE data.yaml
# ------------------------------------------------------------

import yaml

data = {
    "path": str(WORK_DIR),
    "train": "images/train",
    "val": "images/test",
    "nc": len(class_names),
    "names": class_names
}

yaml_path = WORK_DIR / "data.yaml"

with open(yaml_path, "w") as f:
    yaml.dump(
        data,
        f,
        sort_keys=False
    )

print()
print("=" * 70)
print("📄 DATA.YAML")
print("=" * 70)

print("Location:", yaml_path)
print("Classes :", len(class_names))

# ------------------------------------------------------------
# 10. FINAL CHECK
# ------------------------------------------------------------

if (
    len(train_files) == 4000
    and len(test_files) == 2000
    and train_label_count == 4000
    and test_label_count == 2000
    and len(overlap) == 0
):

    print()
    print("🎉 EVERYTHING IS READY!")
    print("✅ 4000 random training images")
    print("✅ 2000 random test images")
    print("✅ 6000 total")
    print("✅ No overlap")
    print("✅ YOLO labels created")
    print("✅ data.yaml created")

else:

    print()
    print("⚠️ PLEASE CHECK THE OUTPUT ABOVE")

🔎 RANDOM SPLIT VERIFICATION
Train images : 4000
Test images  : 2000
Overlap      : 0
✅ NO TRAIN/TEST OVERLAP

📝 LOADING COCO ANNOTATIONS
COCO images      : 6000
COCO annotations : 73602
Classes          : 200

📝 CONVERTING TRAIN ANNOTATIONS
Label files     : 4000
Bounding boxes  : 48895
Missing images  : 0

📝 CONVERTING TEST ANNOTATIONS
Label files     : 2000
Bounding boxes  : 24707
Missing images  : 0

📊 FINAL ANNOTATION CHECK
TRAIN
Images : 4000
Labels : 4000
Boxes  : 48895

TEST
Images : 2000
Labels : 2000
Boxes  : 24707

📄 DATA.YAML
Location: /kaggle/working/rpc_4000_2000/data.yaml
Classes : 200

🎉 EVERYTHING IS READY!
✅ 4000 random training images
✅ 2000 random test images
✅ 6000 total
✅ No overlap
✅ YOLO labels created
✅ data.yaml created


In [8]:
import torch

print("=" * 70)
print("🚀 GPU CHECK")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU count :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)}"
    )

print("=" * 70)

🚀 GPU CHECK
PyTorch : 2.10.0+cu128
CUDA available : True
GPU count : 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [9]:
import yaml

DATA_YAML = "/kaggle/working/rpc_4000_2000/data.yaml"

with open(DATA_YAML, "r") as f:
    data = yaml.safe_load(f)

names = data["names"]

print("=" * 70)
print("🛒 RPC DATASET — 200 PRODUCT CATEGORIES")
print("=" * 70)

for i, name in enumerate(names):
    print(f"{i:3d}: {name}")

print("=" * 70)
print("Total classes:", len(names))

🛒 RPC DATASET — 200 PRODUCT CATEGORIES
  0: 1_puffed_food
  1: 2_puffed_food
  2: 3_puffed_food
  3: 4_puffed_food
  4: 5_puffed_food
  5: 6_puffed_food
  6: 7_puffed_food
  7: 8_puffed_food
  8: 9_puffed_food
  9: 10_puffed_food
 10: 11_puffed_food
 11: 12_puffed_food
 12: 13_dried_fruit
 13: 14_dried_fruit
 14: 15_dried_fruit
 15: 16_dried_fruit
 16: 17_dried_fruit
 17: 18_dried_fruit
 18: 19_dried_fruit
 19: 20_dried_fruit
 20: 21_dried_fruit
 21: 22_dried_food
 22: 23_dried_food
 23: 24_dried_food
 24: 25_dried_food
 25: 26_dried_food
 26: 27_dried_food
 27: 28_dried_food
 28: 29_dried_food
 29: 30_dried_food
 30: 31_instant_drink
 31: 32_instant_drink
 32: 33_instant_drink
 33: 34_instant_drink
 34: 35_instant_drink
 35: 36_instant_drink
 36: 37_instant_drink
 37: 38_instant_drink
 38: 39_instant_drink
 39: 40_instant_drink
 40: 41_instant_drink
 41: 42_instant_noodles
 42: 43_instant_noodles
 43: 44_instant_noodles
 44: 45_instant_noodles
 45: 46_instant_noodles
 46: 47_instant_n

In [10]:
# ============================================================
# 🚀 RPC DATASET - YOLOv8n TRAINING
# 4000 RANDOM TRAIN / 2000 TEST
# 200 PRODUCT CLASSES
# ============================================================

import os
import shutil
import torch
from pathlib import Path

# ------------------------------------------------------------
# 1. INSTALL / CHECK ULTRALYTICS
# ------------------------------------------------------------

try:
    from ultralytics import YOLO
    print("✅ Ultralytics already installed")
except ImportError:
    print("📦 Installing Ultralytics...")
    os.system("pip install -q ultralytics")
    from ultralytics import YOLO
    print("✅ Ultralytics installed")


# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

DATASET = Path("/kaggle/working/rpc_4000_2000")
DATA_YAML = DATASET / "data.yaml"

RUN_DIR = Path("/kaggle/working/yolo_runs")
RUN_NAME = "rpc_4000_yolov8n"

print("=" * 70)
print("📁 DATASET CHECK")
print("=" * 70)

print("Dataset :", DATASET)
print("YAML    :", DATA_YAML)
print("Exists  :", DATA_YAML.exists())


# ------------------------------------------------------------
# 3. VERIFY DATASET
# ------------------------------------------------------------

train_images = DATASET / "images" / "train"
train_labels = DATASET / "labels" / "train"

test_images = DATASET / "images" / "test"
test_labels = DATASET / "labels" / "test"

print("\nTrain images :", len(list(train_images.glob("*"))))
print("Train labels :", len(list(train_labels.glob("*.txt"))))

print("Test images  :", len(list(test_images.glob("*"))))
print("Test labels  :", len(list(test_labels.glob("*.txt"))))


# ------------------------------------------------------------
# 4. GPU CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🚀 GPU CHECK")
print("=" * 70)

print("PyTorch :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("GPU count :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")


# ------------------------------------------------------------
# 5. REMOVE OLD YOLO CACHE FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🧹 CLEANING OLD CACHE")
print("=" * 70)

for cache_file in DATASET.rglob("*.cache"):
    try:
        cache_file.unlink()
        print("Removed:", cache_file)
    except Exception as e:
        print("Could not remove:", cache_file, e)

print("✅ Cache cleanup complete")


# ------------------------------------------------------------
# 6. LOAD YOLOv8 NANO
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🤖 LOADING YOLOv8n")
print("=" * 70)

model = YOLO("yolov8n.pt")

print("✅ YOLOv8n loaded")


# ------------------------------------------------------------
# 7. TRAIN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🔥 STARTING TRAINING")
print("=" * 70)

results = model.train(

    # Dataset
    data=str(DATA_YAML),

    # Model
    pretrained=True,

    # Training
    epochs=30,
    imgsz=640,
    batch=16,

    # IMPORTANT:
    # Use both Tesla T4 GPUs
    device="0,1",

    # Prevent dataset caching from consuming storage
    cache=False,

    # Number of dataloader workers
    workers=2,

    # Validation
    val=True,

    # Save checkpoints
    save=True,
    save_period=5,

    # Early stopping
    patience=10,

    # Augmentation
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,

    # Learning rate
    lr0=0.01,
    lrf=0.01,

    # Output
    project=str(RUN_DIR),
    name=RUN_NAME,

    # Reproducibility
    seed=42,
    deterministic=True,

    # Don't create unnecessary plots/files
    plots=True,

    # Verbose output
    verbose=True
)


# ------------------------------------------------------------
# 8. TRAINING COMPLETE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("🎉 TRAINING COMPLETE")
print("=" * 70)

BEST = RUN_DIR / RUN_NAME / "weights" / "best.pt"
LAST = RUN_DIR / RUN_NAME / "weights" / "last.pt"

print("Best model :", BEST)
print("Exists     :", BEST.exists())

print("Last model :", LAST)
print("Exists     :", LAST.exists())


# ------------------------------------------------------------
# 9. STORAGE CHECK
# ------------------------------------------------------------

def get_size_gb(path):
    total = 0

    if path.is_file():
        return path.stat().st_size / (1024 ** 3)

    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size

    return total / (1024 ** 3)


print("\n" + "=" * 70)
print("💾 STORAGE CHECK")
print("=" * 70)

print(f"Dataset size : {get_size_gb(DATASET):.2f} GB")
print(f"Run size     : {get_size_gb(RUN_DIR):.2f} GB")

usage = shutil.disk_usage("/kaggle/working")

print(f"Used space   : {usage.used / (1024**3):.2f} GB")
print(f"Free space   : {usage.free / (1024**3):.2f} GB")

📦 Installing Ultralytics...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
✅ Ultralytics installed
📁 DATASET CHECK
Dataset : /kaggle/working/rpc_4000_2000
YAML    : /kaggle/working/rpc_4000_2000/data.yaml
Exists  : True

Train images : 4000
Train labels : 4000
Test images  : 2000
Test labels  : 2000

🚀 GPU CHECK
PyTorch : 2.10.0+cu128
CUDA available : True
GPU count : 2
GPU 0: Tesla T4
GPU 1: Tesla T4

🧹 CLEANING OLD CACHE
✅ Cache cleanup complete

🤖 LOADING YOLOv8n
✅ YOLOv8n loaded

🔥 STARTING TR

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/metrics.py:99: UserWarning: Specified kernel cache directory could not be created! This disables kernel caching. Specified directory is /root/.cache/torch/kernels. This warning will appear only once per process. (Triggered internally at /pytorch/aten/src/ATen/native/cuda/jit_utils.cpp:1487.)
  inter = (torch.min(a2, b2) - torch.max(a1, b1)).clamp_(0).prod(2)


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 4.0it/s 15.8s0.2s
                   all       2000      24707    0.00659     0.0907    0.00209     0.0018


/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      2.37G      0.849      4.774      1.085        162        640: 100% ━━━━━━━━━━━━ 250/250 4.8it/s 51.8s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 4.2it/s 15.0s0.3s
                   all       2000      24707     0.0135       0.28     0.0189     0.0158

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/30      2.48G     0.8535      4.321      1.103        144        640: 100% ━━━━━━━━━━━━ 250/250 4.9it/s 51.5s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 4.1it/s 15.3s0.3s
                   all       2000      24707      0.187      0.102     0.0452     0.0381

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/30      2.48G     0.8596        3.9      1.125        103       